### Importações das Bibliotecas Principais

As bibliotecas carregadas acima são fundamentais para o pipeline RAG:

- **RecursiveCharacterTextSplitter**: Divide documentos longos em pequenos pedaços ("chunks") de forma inteligente
- **OpenAIEmbeddings**: Converte texto em vetores numéricos (embeddings) para busca semântica
- **ChatOpenAI**: Interface com o modelo de linguagem (LLM) da OpenAI
- **PyPDFLoader**: Carrega e extrai texto de arquivos PDF
- **Chroma**: Banco de dados vetorial (Vector Store) para armazenar e recuperar embeddings
- **load_qa_chain**: Cria uma cadeia de pergunta-resposta combinando documentos relevantes com o LLM

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [2]:
import os

In [3]:
load_dotenv()

True

### Embeddings e Modelos de Linguagem

**Embeddings** são representações numéricas (vetores) de texto que capturam significado semântico:
- `OpenAIEmbeddings()`: Cria vetores de 1536 dimensões usando o modelo `text-embedding-3-small` da OpenAI
- Permitem buscas por **similaridade semântica** (encontrar documentos com significado parecido, não apenas palavras-chave)

**LLM (Large Language Model)**:
- `model_name="gpt-3.5-turbo"`: Modelo mais rápido e eficiente em custo
- `max_tokens=200`: Limita a resposta a 200 tokens (~150 palavras) para respostas mais concisas

In [4]:
# Carregar os modelos (Embedding e LLM)

embedding_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-3.5-turbo", max_tokens=200)

### Carregamento do Documento PDF

O `PyPDFLoader` faz o seguinte:
- Lê o arquivo PDF página por página
- Extrai o texto de cada página
- Retorna uma lista de documentos (`pages`) onde cada documento representa uma página com seu conteúdo
- `extract_images=False`: Ignora imagens, focando apenas no texto para economizar processamento

Cada página carregada será posteriormente dividida em chunks menores para melhor processamento

In [5]:
# Carregar o PDF

pdf_link = './assets/anexo-projeto-de-lei.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

### Divisão em Chunks (Pedaços Inteligentes)

O `RecursiveCharacterTextSplitter` é crucial para o RAG:

**Por que dividir?** 
- LLMs têm limite de contexto (tokens de entrada)
- Chunks pequenos permitem buscar informações mais precisas

**Parâmetros utilizados:**
- `chunk_size=4000`: Cada chunk tem até 4000 caracteres (~1000 palavras)
- `chunk_overlap=20`: 20 caracteres de sobreposição entre chunks adjacentes (evita perder contexto na divisão)
- `add_start_index=True`: Rastreia a posição original do texto no documento
- `RecursiveCharacterTextSplitter`: Divide por quebras naturais (parágrafos, linhas) mantendo coesão semântica, não apenas no meio de uma frase

O resultado é uma lista de chunks com metadados preservados (página, posição, etc)

In [6]:
# Separar em Chunks ("Pedaços de documento")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

### Visualização dos Chunks

A célula anterior inspeciona a qualidade do chunking:

**Metadados importantes:**
- `page`: Número da página original (0-indexado)
- `page_label`: Rótulo da página no documento (1-indexado)
- `source`: Caminho do arquivo PDF
- `total_pages`: Total de páginas do documento
- `start_index`: Posição do caractere onde o chunk começa
- Informações do PDF: autor, data de criação, software que criou, etc

**O que verificamos:**
- Total de chunks gerados (31 chunks neste caso para 31 páginas)
- Conteúdo dos primeiros chunks (verificar se a divisão faz sentido)
- Integridade dos metadados (confirmando que rastreamento funciona)

In [7]:
# 1. Verificar a quantidade total de pedaços gerados
print(f"Total de chunks gerados: {len(chunks)}")

# 2. Visualizar detalhadamente os 2 primeiros pedaços
for i, chunk in enumerate(chunks[:2]):
    print(f"\n" + "="*50)
    print(f" CHUNK {i+1} ")
    print("="*50)
    
    # Visualizar os Metadados (Página, índice de início, etc)
    print(f"📂 METADADOS: {chunk.metadata}")
    print("-" * 20)
    
    # Visualizar o conteúdo do texto (limitando para não inundar a tela)
    print(f"📄 CONTEÚDO (primeiros 400 caracteres):")
    print(chunk.page_content[:400] + "...")
    print("\n" + "="*50)

Total de chunks gerados: 31

 CHUNK 1 
📂 METADADOS: {'producer': 'Aspose.Words for Java 23.3.0; modified using iText® 5.5.11 ©2000-2017 iText Group NV (AGPL-version)', 'creator': 'Microsoft Office Word', 'creationdate': '2023-05-03T18:22:00+00:00', 'author': 'fredfqd', 'moddate': '2023-05-03T19:15:38-03:00', 'source': './assets/anexo-projeto-de-lei.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'start_index': 0}
--------------------
📄 CONTEÚDO (primeiros 400 caracteres):
PROJETO DE LEI Nº       , DE 2023 
Dispõe sobre o uso da Inteligência Artificial. 
O CONGRESSO NACIONAL decreta: 
CAPÍTULO I 
DISPOSIÇÕES PRELIMINARES 
Art. 1º Esta Lei estabelece normas gerais de caráter nacional para 
o desenvolvimento, implementação e uso responsável de sistemas de 
inteligência artificial (IA) no Brasil, com o objetivo de proteger os direitos 
fundamentais e garantir a impleme...


 CHUNK 2 
📂 METADADOS: {'producer': 'Aspose.Words for Java 23.3.0; modified using iText® 5.5.11 ©2000-2017 iTe

### Armazenamento em Banco de Dados Vetorial (Chroma)

`Chroma` é um Vector Store que armazena embeddings e permite busca por similaridade:

**O que acontece aqui:**
- Cada chunk é convertido em embedding (vetor numérico) usando `OpenAIEmbeddings`
- Os embeddings + metadados são armazenados no Chroma
- `persist_directory="text_index"`: Os dados são salvos em disco para reutilização posterior

**Por que usar Vector Store?**
- Permite busca por similaridade semântica (não apenas por palavras-chave)
- Muito mais rápido que comparar com todos os documentos
- Escalável para grandes volumes de dados
- Recupera os chunks mais relevantes para uma pergunta

O armazenamento é persistente, então não precisa ser recalculado cada vez que executar o notebook

In [8]:
# Salvar no Vector Db - Chroma (Banco de Dados Vetorial)

db = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory="text_index")

### Carregamento do Vector Store e Construção do Pipeline RAG

**Retriever:**
- `vectordb.as_retriever()`: Converte o Vector Store em um retriever
- `search_kwargs={'k': 3}`: Recupera os 3 chunks mais similares para cada pergunta
- Funciona convertendo a pergunta em embedding e buscando os embeddings mais próximos no Chroma

**Chain (Cadeia):**
- `load_qa_chain()`: Cria uma cadeia de pergunta-resposta
- `chain_type='stuff'`: Tipo de cadeia que "stuffs" (coloca) todos os documentos relevantes no contexto do LLM
  - Alternativas: 'map_reduce' (processa cada doc separadamente), 'refine' (refinamento iterativo)
- A chain combina: documentos recuperados + pergunta → enviados ao LLM

Este é o coração do sistema RAG: recuperar + gerar

In [ ]:
# Carregar Db

vectordb = Chroma(persist_directory='text_index', embedding_function=embedding_model)

# Load Retrivier

retriever = vectordb.as_retriever(search_kwargs={ 'k': 3 })

# Construção da cadeia de prompt para chamada do LLM

chain = load_qa_chain(llm, chain_type='stuff')

### Função ask() - O Pipeline RAG Completo

A função `ask()` encapsula o fluxo completo do RAG em 5 passos:

1. **Recuperação**: `retriever.invoke(question)` 
   - Converte pergunta em embedding
   - Busca os 3 chunks mais similares no Chroma
   - Retorna `context` (lista de documentos relevantes)

2. **Construção do prompt**: A pergunta + documentos são formatados
   - Prompt padrão: "Use os documentos abaixo para responder a pergunta"

3. **Invocação da Chain**: `chain.invoke({...})`
   - Envia documentos + pergunta para o LLM (gpt-3.5-turbo)
   - LLM gera resposta contextualizada nos documentos

4. **Resposta**: `resposta_completa['output_text']`
   - Extrai o texto da resposta gerada

5. **Retorno**: Retorna tanto a resposta quanto o contexto (para auditoria)

Este é o fluxo RAG típico: **Retrieve → Augment → Generate**

In [10]:
def ask(question):
    context = retriever.invoke(question)
    
    resposta_completa = chain.invoke({
        'input_documents': context, 
        'question': question 
    })
    
    answer = resposta_completa['output_text']
    return answer, context

### Executando a Pergunta Completa

Abaixo vamos executar a pergunta através do pipeline RAG completo. A pergunta será processada assim:

1. Usuário digita a pergunta
2. `ask()` recupera os 3 chunks mais relevantes do Chroma
3. Esses chunks são enviados como contexto para o LLM
4. O LLM gera uma resposta baseada neste contexto

Para verificar a qualidade do RAG, podemos:
- Ver a resposta gerada
- Inspecionar os chunks recuperados (`context`) para validar se são relevantes
- Confirmar que o LLM respondeu com base nos documentos (não em conhecimento treinado)

In [11]:
user_question = input("User: ")
print(f"User question: {user_question}")

User question: Quais os tópicos criados pela nova lei que minha empresa precisa aplicar?


In [ ]:
answer, context = ask(user_question)
print('Answer: ', answer)

Answer:  Com base no contexto fornecido, os tópicos criados ou abordados pela nova lei que sua empresa precisa aplicar incluem:

1. Medidas para fomentar a inovação em inteligência artificial.
2. Uso de ambiente regulatório experimental (sandbox regulatório).
3. Procedimentos de apuração e critérios de aplicação de sanções administrativas.
4. Autorização de funcionamento de sandboxes regulatórios para inovação.
5. Critérios para solicitações de autorização para sandboxes regulatórios.
6. Parâmetros e critérios para aplicação de sanções após procedimento administrativo.

Esses são os principais tópicos que sua empresa precisará considerar e aplicar de acordo com a nova legislação sobre inteligência artificial.


### Análise dos Contextos Recuperados

As 3 células abaixo mostram os chunks recuperados para a pergunta. Cada chunk contém:

**Estrutura de um contexto:**
- `page_content`: O texto do chunk (corpo da informação)
- `metadata`: Informações sobre origem do chunk
  - `page`: Número da página (0-indexado)
  - `page_label`: Rótulo da página no PDF (1-indexado)
  - `source`: Arquivo de origem
  - Outros: autor, datas, software PDF, etc

**Neste exemplo:**
- **Contexto 1** (página 31/30): Explica medidas de inovação e sandbox regulatório
- **Contexto 2** (página 25/24): Detalha critérios para sandboxes regulatórios
- **Contexto 3** (página 24/23): Especifica parâmetros e critérios de sanções

O LLM usou exatamente estes 3 chunks para gerar a resposta sobre tópicos que empresas precisam aplicar. Isso confirma que o RAG está funcionando corretamente: recuperando e gerando com base em contexto real do documento!

In [13]:
print(context[0])

page_content='São também previstas medidas para fomentar a inovação da 
inteligência artificial, destacando-se o ambiente regulatório experimental 
(sandbox regulatório). 
Com isso, a partir de uma abordagem mista de disposições ex-ante 
e ex-post, a proposição traça critérios para fins de avaliação e desencadeamento 
de quais tipos de ações devem ser tomadas para mitigação dos riscos em jogo, 
envolvendo também os setores interessados no processo regulatório, por meio 
da corregulação. 
Ainda, em linha com o direito internacional, traça balizas para 
conformar direitos autorais e de propriedade intelectual à noção de que os dados 
devem ser um bem comum e, portanto, circularem para o treinamento de 
máquina e o desenvolvimento de sistema de inteligência artificial - sem, 
contudo, implicar em prejuízo aos titulares de tais direitos.  Há, com isso, 
desdobramentos de como a regulação pode fomentar a inovação. Diante do 
exposto, e cientes do desafio que a matéria representa, contamos c

In [14]:
print(context[1])

page_content='§ 3º O disposto neste artigo não substitui a aplicação de sanções 
administrativas, civis ou penais definidas na Lei nº 8.078, de 11 de setembro de 
1990, na Lei nº 13.709, de 14 de agosto de 2018, e em legislação específica. 
§ 4º No caso do desenvolvimento, fornecimento ou utilização de 
sistemas de inteligência artificial de risco excessivo haverá, no mínimo, 
aplicação de multa e, no caso de pessoa jurídica, a suspensão parcial ou total, 
provisória ou definitiva de suas atividades. 
§ 5º A aplicação das sanções previstas neste artigo não exclui, em 
qualquer hipótese, a obrigação da reparação integral do dano causado, nos 
termos do art. 27. 
Art. 37. A autoridade competente definirá, por meio de 
regulamento próprio, o procedimento de apuração e critérios de aplicação das 
sanções administrativas a infrações a esta Lei, que serão objeto de consulta 
pública, sem prejuízo das disposições do Decreto-Lei nº 4.657, de 4 de 
setembro de 1942, Lei nº 9.784, de 29 de janei

In [15]:
print(context[2])

page_content='VI – proibição de tratamento de determinadas bases de dados. 
§ 1º As sanções serão aplicadas após procedimento administrativo 
que possibilite a oportunidade da ampla defesa, de forma gradativa, isolada ou 
cumulativa, de acordo com as peculiaridades do caso concreto e considerados 
os seguintes parâmetros e critérios: 
I – a gravidade e a natureza das infrações e a eventual violação de 
direitos; 
II – a boa-fé do infrator; 
III – a vantagem auferida ou pretendida pelo infrator; 
IV – a condição econômica do infrator; 
V – a reincidência; 
VI – o grau do dano; 
VII – a cooperação do infrator; 
VIII – a adoção reiterada e demonstrada de mecanismos e 
procedimentos internos capazes de minimizar riscos, inclusive a análise de 
impacto algorítmico e efetiva implementação de código de ética; 
IX – a adoção de política de boas práticas e governança; 
X – a pronta adoção de medidas corretivas; 
XI – a proporcionalidade entre a gravidade da falta e a intensidade 
da sanção; e 
